# Fisher Mean-Reversion Backtest Results
Runs the full backtest pipeline and renders performance metrics + charts.

In [ ]:
import asyncio
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd

# Ensure repo root is on the path when running from notebooks/
ROOT = Path().resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.portfolio import Portfolio
from src.order_manager import OrderManager
from src.strategies.fisher_mean_reversion import FisherMeanReversion
from backtest import config
from backtest.execution_engine import BacktestExecutionEngine
from backtest.event_loop import run
from backtest.metrics import compute, summary_df

## Configuration

In [ ]:
STARTING_CAPITAL = 10_000.0
RISK_PER_TRADE   = 0.01       # 1% per trade

# Strategy parameters — tweak these to explore
FISHER_PERIOD    = config.FISHER_PERIOD
FISHER_THRESHOLD = config.FISHER_THRESH
MA_PERIOD        = config.MA_PERIOD

## Load historical data

In [ ]:
df = pd.read_parquet(config.HISTORY_PARQUET)
print(f"Loaded {len(df)} bars  |  {df.index.min().date()} → {df.index.max().date()}")
df.tail(3)

## Run backtest

In [ ]:
strategy    = FisherMeanReversion(
    fisher_period=FISHER_PERIOD,
    fisher_threshold=FISHER_THRESHOLD,
    ma_period=MA_PERIOD,
)
portfolio   = Portfolio(starting_capital=STARTING_CAPITAL)
order_mgr   = OrderManager(risk_per_trade=RISK_PER_TRADE)
engine      = BacktestExecutionEngine(portfolio=portfolio)

portfolio = asyncio.run(run(df, strategy, order_mgr, engine, portfolio))
print(f"Trades completed: {len(portfolio.trade_history)}")

## Performance summary

In [ ]:
metrics = compute(portfolio)
summary_df(metrics)

## Equity curve

In [ ]:
timestamps, equities = zip(*portfolio.equity_curve)
eq = pd.Series(list(equities), index=pd.DatetimeIndex(list(timestamps)))

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(eq.index, eq.values, linewidth=1.5)
ax.axhline(STARTING_CAPITAL, color='grey', linestyle='--', linewidth=0.8, label='Starting capital')
ax.fill_between(eq.index, STARTING_CAPITAL, eq.values,
                where=(eq.values >= STARTING_CAPITAL), alpha=0.15, color='green')
ax.fill_between(eq.index, STARTING_CAPITAL, eq.values,
                where=(eq.values < STARTING_CAPITAL), alpha=0.15, color='red')
ax.set_title('Equity Curve')
ax.set_ylabel('Equity (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend()
plt.tight_layout()
plt.show()

## Drawdown

In [ ]:
peak = eq.cummax()
drawdown = (eq - peak) / peak * 100

fig, ax = plt.subplots(figsize=(12, 3))
ax.fill_between(drawdown.index, drawdown.values, 0, alpha=0.4, color='red')
ax.plot(drawdown.index, drawdown.values, linewidth=0.8, color='red')
ax.set_title('Drawdown (%)')
ax.set_ylabel('Drawdown %')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}%'))
plt.tight_layout()
plt.show()

## P&L distribution

In [ ]:
pnls = [t.pnl for t in portfolio.trade_history]

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['green' if p > 0 else 'red' for p in pnls]
ax.bar(range(len(pnls)), pnls, color=colors, alpha=0.7, width=0.8)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('P&L per Trade')
ax.set_xlabel('Trade #')
ax.set_ylabel('P&L (USD)')
plt.tight_layout()
plt.show()

## Exit reason breakdown

In [ ]:
trade_df = portfolio.to_dataframe()
trade_df.groupby('exit_reason').agg(
    count=('pnl', 'count'),
    avg_pnl=('pnl', 'mean'),
    total_pnl=('pnl', 'sum'),
    win_rate=('pnl', lambda x: (x > 0).mean() * 100),
).round(2)

## Full trade log

In [ ]:
trade_df.sort_values('closed_at').style.applymap(
    lambda v: 'color: green' if isinstance(v, float) and v > 0 else
              'color: red'   if isinstance(v, float) and v < 0 else '',
    subset=['pnl']
)